#### General Implementation of the Levenberg-Marquardt Algorithm for Nonlinear Least-Squares Problems 

From Chapter 10 of the textbook, a nonlinear least-squares problem has the special objective 
<br>
\
$
f(x) = \frac{1}{2} \| r(x) \|_2^2 = \frac{1}{2} \sum_{i=1}^m r_i(x)^2,
$ 
<br>
\
where each residual $ $r_i(x) : \mathbb{R}^n \to \mathbb{R}$ $ is a smooth funtion (typically $ m \geq n $). The gradient and (approximate) Hessian of $f$ are
<br>
\
$
\nabla f(x) = J(x)^T r(x), \qquad \nabla^2 f(x) \approx J(x)^T J(x),
$ 
<br>
\
with $J(x)$ the $m \times n$ **Jacobian matrix** of the residuals.


**Gauss-Newton** is the natual extention of the linear least-squares normal equations. it solves the linear subproblem
<br>
\
$J_k^T J_k \, p_k^{GN} = -J_k^T r_k$
\
\
at each iteration $x_k$, then takes $x_{k+1} + p_k^{GN}$. This works well near a solution when the residuals are small or the second-order term $\sum r_i \nabla^2 r_i$ is negligible.

**Levenberg-Marquardt (LM)** improves robustness by adding a damping (trust-region) term. Instead of the plain Gauss-Newton step, it solves

$(J_k^T J_k + \lambda_k I) \, p_k = -J_k^T r_k, \qquad \lambda_k > 0$.

This is exactly the normal equations of the linear least-squares subproblem

$\min_p \quad \frac{1}{2} \| J_k p + r_k \|^2 \quad \text{subject to} \quad \| p \| \leq \Delta_k$.

- As $\lambda_k \to 0$, the step approaches the Gauss-Newton direction.
- As $\lambda_k \to \infty$, the step behaves like a steepest-descent step scaled by $1/\lambda_k$.

#### General LM algoritm - Classic implementation

1. Start with an initial guess $x_0$, initial damping $\lambda_0 > 0$, and a trust-region radius $\Delta_0$
2. At iteration $k$:
    - Compute the residual vector $r_k = r(x_k)$ and Jacobian $J_k = J(x_k)$.
    - Solve the damped normal equations for the trial step $p$.
    - Compute the trial point $x^+ = x_k + p$ and the new residual norm.
    - **Acceptance test**: If $\|r(x^{k+1})\| < \|r(x_k)\|$, accept the step, set $x_{k+1} = x^{k+1}$, and **decrease** $\lambda$ (or increase $\Delta$).
    Otherwise reject and **increase** $\lambda$ (or shrink $\Delta$).
3. Stop when convergence criteria are met (small $\|J^T r\|$, small step, or maximum iterations reached).

#### Theoratical properties of the Levenberg-Marquardt Algorithm

The Levenberg-Marquardt algorithm can be interpreted as a trust-region method that solves, at each iteration $k$, the contrained linear least-squares subproblem

$\min_p \quad \frac{1}{2} \| J_k p + r_k \|^2 \quad \text{subject to} \quad \| p \| \leq \Delta_k,$

where $r_k = r(x_k)$ and $J_k = J(x_k)$. This subproblem is equivalent to the damped normal equations

$(J_k^T J_k + \lambda_k I) p_k = -J_k^T r_k, \qquad \lambda_k > 0.$

The damping parameter $\lambda_k$ is adjusted dynamically: it is decreased after successful steps and increased after unsuccessful steps.

##### Global Convergence

Under standard assumptions (each residual $r_j$) is Lipschitz continuously differentiable and the Jacobians satisfy a uniform full-rank condition in a neighborhood of the bounded level set $\mathcal{L} = \{ x \mid f(x) \leq f(x_0) \}$, the LM algorithm is guaranteed to converge globally to a stationary point. A theorem (10.3) states that

$\lim_{k \to \infty} \nabla f(x_k) = 0.$

This guarantee holds even when the residuals are large or the Gauss-Newton approximation is inaccurate, thanks to the damping mechanism.

##### Local Convergence Rate

Near a solution $x^*$ where the residuals are small ($\| r(x^*) \| \approx 0$) and $J(x^*)$ has full column rank, the second-order term in the Hessian becomes negligible. In this regime, LM behaves identically to the pure Gauss-Newton method and exhibits **quadratic local convergence**. This rapid final-stage convergence is clearly visible in our implementation, where the objective value drops from over 560 to approximately 0.25 within roughly 10–12 iterations once the iterate enters the vicinity of the solution.

##### Computational Complexity
For a dense Jacobian of size $m \times n$ ($m \geq n$), each iteration has the following dominant costs:
- Forming $J_k^T J_k$ and $J_k^T r_k$: $O(m n^2)$
- Solving the $n \times n$ damped system via Cholesky factorization: $O(n^3)$

Thus, the per-iteration complexity is $O(m n^2)$. In our notebook ($m=8$, $n=3$), this cost is negligible, but the structure scales efficiently to moderate-sized nonlinear least-squares problems common in data fitting and parameter estimation.

##### Advantages over Pure Gauss-Newton
The damping term $\lambda_k I$ ensures that the coefficient matrix $J_k^T J_k + \lambda_k I$ is always positive definite, even when $J_k$ is rank-deficient. This prevents the algorithm from failing or diverging when far from the solution — a common weakness of the plain Gauss-Newton method. By automatically transitioning from gradient-descent-like behavior (large $\lambda_k$) to Gauss-Newton behavior (small $\lambda_k$), LM combines global robustness with fast local convergence.